## Scenario 2

An e-commerce company receives customer registration data from multiple sources. Some customer records contain missing mandatory information due to:</br>

- Customers skipping fields</br>
- Source system issues</br>
- Incomplete file uploads</br>

The data engineering team must clean customer data and validate that mandatory fields are available.

### 1.0 Objective

- Detect missing customer values.
- Standardize missing values.
- Remove invalid customer records based on business rules.
- Validate cleaned customer data.

### 1.1 Business Rules</br>
#### 1.1.1 Email Rules
- Email cannot be NULL.
- Email cannot be empty.
- Email cannot contain only spaces.
- Email cannot contain placeholder values.
- Email should not contain leading\trailing spaces.

#### 1.1.2 Phone Rules
- Phone cannot be NULL.
- Phone cannot be empty.
- Phone cannot contain only spaces.
- Phone cannot contain placeholder values such as "NULL", "N/A", "NA", "NaN", "NONE", "UNKNOWN".

### 1.2 Cleaning Steps
- Remove leading and trailing spaces from string columns.
- Convert empty values ("") into NULL.
- Convert blank values (spaces only) into NULL.
- Convert placeholder values ("NULL", "N/A", "NaN", "NONE") into NULL.
- Standardize missing values into a common format.

### 1.3 Validation
- Check mandatory fields after cleaning.
- Validate that Email and Phone fields are available.
- Identify valid and invalid customer records.
- Separate valid records and rejected records.

### 2.0 Create Sample Customer Data


In [0]:
%sql
CREATE TABLE Customers (
    CustomerID INT,
    CustomerName VARCHAR(100),
    Email VARCHAR(255),
    Phone VARCHAR(20)
);


In [0]:
%sql
INSERT INTO Customers (CustomerID, CustomerName, Email, Phone)
VALUES
    (1001, "Alpha", "alpha@test.com", "9110110162"),
    (1002, "Bravo", NULL, "8005183005"),              -- Email is NULL
    (1003, "Emma", "", "7208957715"),                 -- Email is empty
    (1004, "Robert", "   ", "9873654528"),           --  Email contains spaces
    (1005, "Sarah", "sarah@test.com", NULL),          -- Phone is NULL
    (1006, "Michael", "michael@test.com", "9895667234"),
    (1007, "Charley", NULL, NULL),                      -- Email and Phone are NULL
    (1008, "Linda", "", "9642624219"),                -- Empty Email
    (1009, "William", "william@test.com", ""),        -- Empty Phone
    (1010, "Sophia", "   ", "8712546930"),            -- Space-only Email
    (1011, "Alice",   "N/A",   "9898545412"),   -- Placeholder text value: "N/A" in Email
    (1012, "Bob",     "NaN",   "9110110812"),   -- Placeholder text value: "NaN" in Email
    (1013, "Zach", "NULL",  "N/A");             -- Placeholder text values: "NULL" in Email, "N/A" in Phone

num_affected_rows,num_inserted_rows
13,13


### 2.1 Display Customer Source Data

In [0]:
%sql
-- Display customer records before applying data cleaning rules
SELECT * FROM customers;

CustomerID,CustomerName,Email,Phone
1001,Alpha,alpha@test.com,9110110162
1002,Bravo,null,8005183005
1003,Emma,,7208957715
1004,Robert,,9873654528
1005,Sarah,sarah@test.com,null
1006,Michael,michael@test.com,9895667234
1007,Charley,null,null
1008,Linda,,9642624219
1009,William,william@test.com,
1010,Sophia,,8712546930


### 3.0 Data Profiling

#### 3.1 Find NULL values

In [0]:
%sql
SELECT * FROM customers 
WHERE Email IS NULL OR Phone IS NULL; 

CustomerID,CustomerName,Email,Phone
1002,Bravo,null,8005183005
1005,Sarah,sarah@test.com,null
1007,Charley,null,null


#### 3.2 Find Empty String Values

In [0]:
%sql
SELECT * FROM customers 
WHERE Email = "" OR Phone = "";

CustomerID,CustomerName,Email,Phone
1003,Emma,,7208957715
1008,Linda,,9642624219
1009,William,william@test.com,


#### 3.3 Find Blank Values 

In [0]:
%sql
SELECT * FROM customers 
WHERE TRIM(Email) = "" OR TRIM(Phone) = "";

CustomerID,CustomerName,Email,Phone
1003,Emma,,7208957715
1004,Robert,,9873654528
1008,Linda,,9642624219
1009,William,william@test.com,
1010,Sophia,,8712546930


#### 3.4 Find Place Holder Values

In [0]:
%sql
SELECT * FROM customers
WHERE LOWER(Email) IN ('n/a', 'na', 'null', 'unknown')
OR LOWER(Phone) IN ('n/a', 'na', 'null', 'unknown');

CustomerID,CustomerName,Email,Phone
1011,Alice,N/A,9898545412
1013,Zach,NULL,N/A


### 4.0 Data Cleaning

#### 4.1 Trim Leading and Trailing Spaces 

In [0]:
%sql
SELECT CustomerID, CustomerName, 
TRIM(Email) AS Email,
TRIM(Phone) AS Phone
FROM Customers;

CustomerID,CustomerName,Email,Phone
1001,Alpha,alpha@test.com,9110110162
1002,Bravo,null,8005183005
1003,Emma,,7208957715
1004,Robert,,9873654528
1005,Sarah,sarah@test.com,null
1006,Michael,michael@test.com,9895667234
1007,Charley,null,null
1008,Linda,,9642624219
1009,William,william@test.com,
1010,Sophia,,8712546930


#### 4.2 Convert Empty Values to NULL

In [0]:
%sql
SELECT CustomerID, CustomerName,
NULLIF(Email, '') AS Email,
NULLIF(Phone, '') AS Phone
FROM customers;

CustomerID,CustomerName,Email,Phone
1001,Alpha,alpha@test.com,9110110162
1002,Bravo,null,8005183005
1003,Emma,null,7208957715
1004,Robert,,9873654528
1005,Sarah,sarah@test.com,null
1006,Michael,michael@test.com,9895667234
1007,Charley,null,null
1008,Linda,null,9642624219
1009,William,william@test.com,null
1010,Sophia,,8712546930


#### 4.3 Convert Blank Values (only Spaces) to NULL

In [0]:
%sql
SELECT CustomerID, CustomerName,
CASE 
    WHEN TRIM(Email) = '' THEN NULL ELSE Email END AS Email,
CASE 
    WHEN TRIM(Phone) = '' THEN NULL ELSE Phone END AS Phone
FROM customers;

CustomerID,CustomerName,Email,Phone
1001,Alpha,alpha@test.com,9110110162
1002,Bravo,null,8005183005
1003,Emma,null,7208957715
1004,Robert,null,9873654528
1005,Sarah,sarah@test.com,null
1006,Michael,michael@test.com,9895667234
1007,Charley,null,null
1008,Linda,null,9642624219
1009,William,william@test.com,null
1010,Sophia,null,8712546930


#### 4.4 Convert Place Holder Values to NULL

In [0]:
%sql
SELECT CustomerID, CustomerName,
CASE 
    WHEN TRIM(Email) = '' OR LOWER(TRIM(Email)) IN ('null', 'n/a', 'na', 'nan', 'none', 'unknown')
    THEN NULL 
    ELSE Email END AS Email,

CASE 
    WHEN TRIM(Phone) = '' OR LOWER(TRIM(Phone)) IN ('null', 'n/a', 'na', 'nan', 'none', 'unknown')
    THEN NULL
    ELSE Phone END AS Phone

FROM customers;

CustomerID,CustomerName,Email,Phone
1001,Alpha,alpha@test.com,9110110162
1002,Bravo,null,8005183005
1003,Emma,null,7208957715
1004,Robert,null,9873654528
1005,Sarah,sarah@test.com,null
1006,Michael,michael@test.com,9895667234
1007,Charley,null,null
1008,Linda,null,9642624219
1009,William,william@test.com,null
1010,Sophia,null,8712546930


### 5.0 Create Cleaned View

In [0]:
%sql
CREATE VIEW customers_cleaned AS
SELECT CustomerID, CustomerName, 
CASE
    WHEN TRIM(Email) = '' OR LOWER(TRIM(Email)) IN ('null','n/a','na','nan','none','unknown')
    THEN NULL 
    ELSE TRIM(Email) END AS Email,
CASE
    WHEN TRIM(Phone) = '' OR LOWER(TRIM(Phone)) IN ('null','n/a','na','nan','none','unknown')
    THEN NULL
    ELSE TRIM(Phone) END AS Phone
FROM Customers;

### 6.0 Data Validation

#### 6.1 Valid Customer Records

In [0]:
%sql
SELECT * FROM customers_cleaned
WHERE Email IS NOT NULL
AND Phone IS NOT NULL;

CustomerID,CustomerName,Email,Phone
1001,Alpha,alpha@test.com,9110110162
1006,Michael,michael@test.com,9895667234


#### 6.2 Rejected Customer Records

In [0]:
%sql
SELECT * FROM customers_cleaned
WHERE Email IS NULL
OR Phone IS NULL;

CustomerID,CustomerName,Email,Phone
1002,Bravo,null,8005183005
1003,Emma,null,7208957715
1004,Robert,null,9873654528
1005,Sarah,sarah@test.com,null
1007,Charley,null,null
1008,Linda,null,9642624219
1009,William,william@test.com,null
1010,Sophia,null,8712546930
1011,Alice,null,9898545412
1012,Bob,null,9110110812


### 6.3 Validation Results

- Valid Customer Records Count: 2
- Rejected Customer Records Count: 11

### 7.0 Key Learnings

- Identified missing and inconsistent values through data profiling.
- Cleaned and standardized empty, blank, and placeholder values.
- Applied Email and Phone validation rules based on business requirements.
- Performed a customer data cleaning and validation workflow using SQL.
- Improved customer data quality through data cleaning and validation.